In [ ]:
# Copyright 2024 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

## What you'll need

* A Google Cloud Account and Google Cloud Project

## Basic Setup
### Install dependencies

In [ ]:
%pip install \
    google-cloud-alloydb-connector[asyncpg]==1.4.0 \
    sqlalchemy==2.0.36 \
    pandas==2.2.2 \
    vertexai==1.70.0 \
    asyncio==3.4.3 \
    greenlet==3.1.1 \
    langchain_google_alloydb_pg==0.9.3 \
    datasets==3.4.0 \
    google-genai==0.3.0 \
    --quiet

### Authenticate to Google Cloud within Colab
If you're running this on google colab notebook, you will need to Authenticate as an IAM user.

In [ ]:
from google.colab import auth

auth.authenticate_user()

### Define Notebook Parameters

In [ ]:
# @markdown Update the parameters below to match your environment.

# Please fill in these values.
region = "my-region"  # @param {type:"string"}
cluster = "my-cluster"  # @param {type:"string"}
instance = "my-instance"  # @param {type:"string"}
database = "my-database"  # @param {type:"string"}
table = "video_embeddings"   # @param {type:"string"}
bucket = "my-bucket"  # @param {type:"string"}
embeddings_table_name = "video_embeddings"   # @param {type:"string"}
password = input("Please provide a password to be used for 'postgres' database user: ")

### Connect Your Google Cloud Project

In [ ]:
# Configure gcloud.
!gcloud config set project {project_id}

### Enable APIs for AlloyDB and Vertex AI

You will need to enable these APIs in order to create an AlloyDB database and utilize Vertex AI as an embeddings service!

In [ ]:
!gcloud services enable alloydb.googleapis.com aiplatform.googleapis.com

### Configure Logging

In [ ]:
import logging
import sys

# Configure the root logger to output messages with INFO level or above
logging.basicConfig(level=logging.INFO, stream=sys.stdout, format='%(asctime)s[%(levelname)5s][%(name)14s] - %(message)s',  datefmt='%H:%M:%S', force=True)

### Initialize GenAI Client

In [ ]:
from google import genai
from google.genai import types

genai_client = genai.Client(
    vertexai=True, project=project_id, location=region
)

### Connect to AlloyDB
You will need a Postgres AlloyDB instance for the following stages of this notebook. Create one now if you have not already created it.

> NOTE: You will need to install or upgrade pgvector version 0.7.0 or later to use sparse embeddings in this notebook.
>
> Example command to upgrade pgvector version: `ALTER EXTENSION vector UPDATE TO '0.7.4.google-1';`

This function will create a connection pool to your AlloyDB instance using the AlloyDB Python connector. The AlloyDB Python connector will automatically create secure connections to your AlloyDB instance using mTLS.

In [ ]:
import asyncpg

import sqlalchemy
from sqlalchemy.ext.asyncio import AsyncEngine, create_async_engine

from google.cloud.alloydb.connector import AsyncConnector, IPTypes

async def init_connection_pool(connector: AsyncConnector, db_name: str, pool_size: int = 5) -> AsyncEngine:
    # initialize Connector object for connections to AlloyDB
    connection_string = f"projects/{project_id}/locations/{region}/clusters/{cluster}/instances/{instance}"

    async def getconn() -> asyncpg.Connection:
        conn: asyncpg.Connection = await connector.connect(
            connection_string,
            "asyncpg",
            user="postgres",
            password=password,
            db=database,
            ip_type=IPTypes.PRIVATE,
        )
        return conn

    pool = create_async_engine(
        "postgresql+asyncpg://",
        async_creator=getconn,
        pool_size=pool_size,
        max_overflow=0,
    )
    return pool

connector = AsyncConnector()

# Instantiate re-useable pool for use throughout the notebook.
global pool
pool = await init_connection_pool(connector, "postgres")

### Create AlloyDB Query Helper Function

In [ ]:
# Create AlloyDB Query Helper Function
from sqlalchemy import text, exc
import pandas as pd

async def run_query(sql):
  """Executes a SQL query against the AlloyDB database.

  This function accepts a SQL string and performs the following actions:
  - If the SQL statement starts with 'SELECT' or 'WITH' (case-insensitive),
    it executes the query and returns the results as a Pandas DataFrame
    with column names derived from the query.
  - For other types of SQL statements (e.g., INSERT, UPDATE, DELETE),
    it executes the query and returns the SQLAlchemy ResultProxy object
    after committing the transaction.

  Args:
    sql: A string containing the SQL query to execute.

  Returns:
    pandas.DataFrame: If the SQL statement is a SELECT or WITH query,
      a DataFrame containing the query results.
    sqlalchemy.engine.result.ResultProxy: If the SQL statement is not a
      SELECT or WITH query, a ResultProxy object representing the
      result of the execution.
    None: If a `sqlalchemy.exc.ProgrammingError` occurs during query execution,
      the error is printed to the console, and None is returned.

  Raises:
    sqlalchemy.exc.ProgrammingError: If there is an issue with the SQL syntax
      or the database operation. The error is caught, printed, and None is
      returned.

  Example Usage:
  >>> # SELECT query
  >>> sql_select = "SELECT ticker, company_name from investments LIMIT 5"
  >>> df_result = await run_query(sql_select)
  >>> print(df_result)
  >>>
  >>> # INSERT query
  >>> sql_insert = "INSERT INTO investments (ticker, company_name) VALUES ('NEW', 'New Company')"
  >>> insert_result = await run_query(sql_insert)
  >>> print(insert_result) # Output will be the ResultProxy object
  """
  async with pool.connect() as conn:
    if sql.strip().lower().startswith('select') or sql.strip().lower().startswith('with'):
      try:
        result = await conn.execute(sqlalchemy.text(sql))
        rows = result.fetchall()
        column_names = result.keys()
        df = pd.DataFrame(rows, columns=column_names)
        return df
      except exc.ProgrammingError as e:
        print(e)
    else:
      try:
        result = await conn.execute(
            text(sql)
        )
        await conn.commit()
        operation_type = sql.split()[0].upper()
        row_count = result.rowcount
        if operation_type in ['INSERT', 'UPDATE', 'DELETE']:
          print(f"{operation_type} statement executed successfully. {row_count} row(s) affected.")
        else:
          print(f"{operation_type} statement executed successfully.")
        return result

      except exc.ProgrammingError as e:
        print(e)



## Import Sample Videos

### Create or Instantiate Bucket

In [ ]:
# Create bucket if none is provided
import uuid
from google.cloud import storage

storage_client = storage.Client()
random_suffix = uuid.uuid4().hex[:6]  # Get a 6-character hexadecimal suffix
bucket_name = f"test-videos-{random_suffix}"

if not bucket:
    bucket = storage_client.create_bucket(bucket_name)
    print(f"Bucket {bucket.name} created.")
else:
    print(f"Using provided bucket: {bucket}")
    bucket = storage_client.bucket(bucket)

### Get Sample Videos from Huggingface

In [ ]:
# Login to Huggingface
!huggingface-cli login

In [ ]:
# Download Huggingface repo snapshot with video files
from huggingface_hub import snapshot_download
local_path = snapshot_download(repo_id="yaak-ai/lerobot-driving-school", repo_type="dataset")
print(f"Repo saved to: {local_path}")

### Upload Sample Videos to GCS

In [ ]:
# Upload test videos to GCS so that the embedding model can access them.

import os
import uuid
from google.cloud import storage

"""Recursively finds and uploads video files to a GCS bucket."""

video_dir = os.path.join(local_path, "videos", "chunk-000")

gcs_uris = []

if not os.path.exists(video_dir):
    raise(f"Directory {video_dir} does not exist.")

for root, _, files in os.walk(video_dir):
    for file in files:
        if file.lower().endswith(('.mp4', '.avi', '.mov', '.mkv')):  # Add more video extensions if needed
            local_file_path = os.path.join(root, file)
            relative_path = os.path.relpath(local_file_path, video_dir) #get the relative path to keep the directory structure in GCS.
            blob_path = relative_path.replace(os.sep, '/') #replace windows backslash with forward slash for GCS.
            blob = bucket.blob(blob_path)

            blob.upload_from_filename(local_file_path)
            gcs_uri = f"gs://{bucket.name}/{blob.name}"
            gcs_uris.append(gcs_uri)
            print(f"Uploaded {local_file_path} to {gcs_uri}")

print(f"Done uploading {len(gcs_uris)} files.")


## Generate Embeddings


### Define Embedding Functions

In [ ]:
import asyncio
import concurrent.futures
from typing import Union, List, AsyncIterator, Any
import vertexai
from vertexai.vision_models import MultiModalEmbeddingModel, Video
from vertexai.vision_models import VideoSegmentConfig
from google.api_core.exceptions import ResourceExhausted


async def embed_video(
    video_uri: str,
    model: MultiModalEmbeddingModel,
    retries: int = 100,
    delay: int = 30,
) -> List[Any]:

    logger = logging.getLogger("embed_video")

    # Retry loop
    for attempt in range(retries):
        try:
            # Get descriptive text
            text_prompt = "Please watch this video and provide a 1-sentence summary of what you see."

            logger.info(f"Generating description for video {video_uri}")
            text_description = await genai_client.aio.models.generate_content(
                model='gemini-2.0-flash',
                contents=[
                    types.Part.from_text(text_prompt),
                    types.Part.from_uri(video_uri, 'video/mp4')
                ]
            )

            # Get video embeddings
            logger.info(f"Generating embeddings for video {video_uri}")
            embeddings = model.get_embeddings(
                video=Video.load_from_file(video_uri),
                video_segment_config=VideoSegmentConfig(interval_sec=10),
                contextual_text=text_description.text

            logger.info(f"Kicked off embeddings function for video {video_uri}")
            #logger.info(f"Generated embeddings: {embeddings}")
            return {'embedding_object': embeddings, 'video_description': text_description.text}

        except Exception as e:
            if attempt < retries - 1:  # Retry only if attempts are left
                logger.warning(f"Error: {e}. Retrying in {delay} seconds...")
                await asyncio.sleep(delay)  # Wait before retrying
            else:
                logger.error(f"Failed to get embeddings for video: {video_uri} after {retries} attempts.")
    return []


async def embed_videos_concurrently(
    video_uris: List[str],
    model: MultiModalEmbeddingModel,
    max_concurrency: int = 5,
) -> AsyncIterator[List[dict[str, Union[str, List[float]]]]]:

    logger = logging.getLogger("embed_videos_concurrently")
    queue = asyncio.Queue()
    for uri in video_uris:
        await queue.put(uri)

    tasks = {}  # Use a dictionary to store tasks and their URIs
    while not queue.empty() or tasks:
        # Wait for at least one task to complete *if* tasks exist
        if tasks:
            done, _ = await asyncio.wait(tasks, return_when=asyncio.FIRST_COMPLETED)
            logger.info(f"Looping through {len(done)} tasks...")
            for task in done:
                video_uri = tasks.pop(task) # Get the URI *and* remove the task.
                logger.info(f"Awaiting task...")
                result = await task  # Await the task to get the result
                if result:
                    logger.info(f"Embedding task completed: Processed video {video_uri}.")
                    yield (video_uri, result)

        # Calculate how many new tasks to add
        num_to_add = min(max_concurrency - len(tasks), queue.qsize())

        # Create tasks up to the concurrency limit, AFTER handling completed tasks
        for _ in range(num_to_add):
            video_uri = await queue.get()
            task = asyncio.create_task(embed_video(video_uri, model))
            logger.info(f"Adding task for {video_uri}...")
            tasks[task] = video_uri  # Store the URI *with* the task


### Run the Embeddings Workflow

In [ ]:
import time
import vertexai
from vertexai.vision_models import MultiModalEmbeddingModel, Video
from vertexai.vision_models import VideoSegmentConfig
from datetime import datetime, timezone

vertexai.init(project=project_id, location=region)

embeddings_table_name = "video_embeddings3"


# Model to use for generating embeddings
model_name = "multimodalembedding@001"

### Embeddings workflow ###


async def run_embeddings_workflow(
    video_uris: List[str] = gcs_uris,
    embed_video_concurrency: int = 2
):
    # Establish to AlloyDB engine client
    from langchain_google_alloydb_pg import AlloyDBEngine
    from google.cloud.alloydb.connector import IPTypes

    alloydb_engine = await AlloyDBEngine.afrom_instance(
        project_id=project_id,
        region=region,
        cluster=cluster,
        instance=instance,
        database=database,
        user='postgres',
        password=password,
        ip_type=IPTypes.PRIVATE,
    )
    print("Langchain AlloyDB client initiated.")

    # Initialize VertexAI and the model to be used to generate embeddings
    vertexai.init(project=project_id, location=region)
    model = MultiModalEmbeddingModel.from_pretrained(model_name)
    print(f"Vertex AI model {model_name} initialized.")

    # Initialize Vector Store Table in AlloyDB for
    from langchain_google_alloydb_pg import Column

    await alloydb_engine.ainit_vectorstore_table(
            table_name=embeddings_table_name,
            vector_size=1408,
            overwrite_existing=True,
            embedding_column='embedding',
            id_column=Column("id", "uuid"),  #  Default is Column("langchain_id", "UUID")
            metadata_columns=[
                Column("uri","text"),
                Column("embedding_type","text"),
                Column("embedding_model","text"),
                ],
            metadata_json_column="metadata_json",
        )
    print("AlloyDB vector store table initiated for dense embeddings.")

    # Connect instance of AlloyDBVectorStore to AlloyDB Vector Store Table
    from langchain_google_alloydb_pg import AlloyDBVectorStore

    vs_dense = await AlloyDBVectorStore.create(
            engine=alloydb_engine,
            embedding_service=model,
            table_name=embeddings_table_name,
            embedding_column='embedding',
            id_column="id",  # Must match id_column defined in ainit_vectorstore_table()
            metadata_columns=[
                "uri",
                "embedding_type",
                "embedding_model",
                ],
            metadata_json_column="metadata_json",
        )
    print("Connected to AlloyDB vector store.")

    start_time = time.monotonic()

    # Embed videos and update the database concurrently
    async for current_video_uri, embeddings_batch in embed_videos_concurrently(
        video_uris,
        model,
        max_concurrency=embed_video_concurrency
    ):
        if embeddings_batch:  # Ensure we have embeddings
            #Prepare embeddings and texts to be added to the database
            ids = []
            texts = []
            embeddings = []
            metadatas = []

            # The video embeddings are a list of embeddings, one per segment.
            for index, embedding in enumerate(embeddings_batch['embedding_object'].video_embeddings):
                # Format: uri:start_offset:end_offset
                texts.append(f"{current_video_uri}:{embedding.start_offset_sec}:{embedding.end_offset_sec}")
                embeddings.append(embedding.embedding)
                # Add some metadata.
                metadatas.append({"date_created": str(datetime.now(timezone.utc)),
                                  "uri": current_video_uri,
                                  "start_offset_sec": embedding.start_offset_sec,
                                  "end_offset_sec": embedding.end_offset_sec,
                                  "embedding_type": "video",
                                  "embedding_model": model_name,

                                  })
                ids.append(str(uuid.uuid4())) #generate an id

            # The text embedding is one embedding per video segment
            texts.append(embeddings_batch['video_description'])
            embeddings.append(embeddings_batch['embedding_object'].text_embedding)
            metadatas.append({"date_created": str(datetime.now(timezone.utc)),
                                  "uri": current_video_uri,
                                  "embedding_type": "text",
                                  "embedding_model": model_name,
                                  })
            ids.append(str(uuid.uuid4()))

            #Add to database
            logging.info(f"Adding embeddings to database. Number of embeddings: {len(embeddings)}") # useful for checking
            await vs_dense.aadd_embeddings(
                ids=ids,
                texts=texts,
                embeddings=embeddings,
                metadatas=metadatas
            )


    end_time = time.monotonic()
    elapsed_time = end_time - start_time

    # Release database connections and close the connector

    print(f"Job started at: {time.ctime(start_time)}")
    print(f"Job ended at: {time.ctime(end_time)}")
    print(f"Total run time: {elapsed_time:.2f} seconds")


await run_embeddings_workflow()

## Create a ScaNN Index for Efficient ANN Vector Search

In [ ]:
sql_array = []

sql_array.append("CREATE EXTENSION IF NOT EXISTS alloydb_scann")
sql_array.append("SET SESSION scann.num_leaves_to_search = 1")
sql_array.append("SET SESSION scann.pre_reordering_num_neighbors=50")
sql_array.append("DROP INDEX IF EXISTS embeddings_scann_2")
sql_array.append("""
CREATE INDEX embeddings_scann_2 ON video_embeddings
  USING scann (embedding cosine)
  WITH (num_leaves=2);
""")

for s in sql_array:
  await run_query(s)


## Query the Embeddings

### Search with Vector Embeddings (i.e. Dense Search)

In [ ]:
# Get a sample query embedding
sql = """
SELECT * FROM video_embeddings LIMIT 1
"""

result = await run_query(sql)
query_embedding = result.embedding[0]
result

In [ ]:
# Find similar video clips and include rank
sql = f"""
WITH vector_search AS (
  SELECT embedding <=> '{query_embedding}'::vector AS distance, *
    FROM video_embeddings
  ORDER BY distance
  LIMIT 10
) SELECT RANK () OVER (ORDER BY distance) AS vector_rank, *
FROM vector_search
"""

result = await run_query(sql)
result

### Search with Fulltext Search (i.e. Sparse Search)

In [ ]:
# Create FTS GIN Index
# Ref: https://www.postgresql.org/docs/current/gin.html
sql = "CREATE INDEX ON video_embeddings USING GIN (to_tsvector('english', content))"

await run_query(sql)

In [ ]:
# Query Based on FTS
# Ref: https://www.postgresql.org/docs/current/textsearch-controls.html
fts_search_token = "car & snow"

sql = f"""
WITH fts_search AS (
  SELECT
      ts_rank(to_tsvector('english', content), to_tsquery('{fts_search_token}')) AS ts_rank_score,  *
  FROM video_embeddings
  WHERE to_tsvector('english', content) @@ to_tsquery('{fts_search_token}')
  ORDER BY ts_rank_score desc
  LIMIT 10
) SELECT
  RANK () OVER (ORDER BY ts_rank_score desc) AS fts_rank, *
  FROM fts_search
"""

result = await run_query(sql)
result

### Search with Hybrid Search (i.e. Dense + Sparse Search with Re-ranking)

In [ ]:
# Get a sample query embedding
sql = """
SELECT * FROM video_embeddings LIMIT 1
"""
result = await run_query(sql)
query_embedding = result.embedding[0]

# Define FTS token
fts_search_token = "car & snow"

# Set top_k
top_k = 10

# Define RRF smoothing constant k
rrf_k = 60

# Run a hybrid search combining vector search and fulltext search
# Reciprocal Rank Fusion implemented as rrf_score below
sql = f"""
WITH vector_search AS (
  SELECT embedding <=> '{query_embedding}'::vector AS distance,
    RANK () OVER (ORDER BY embedding <=> '{query_embedding}'::vector) AS vector_rank, *
    FROM video_embeddings
  ORDER BY distance
  LIMIT {top_k * 2}
), fts_search AS (
  SELECT
      ts_rank(to_tsvector('english', content), to_tsquery('{fts_search_token}')) AS ts_rank_score,
      RANK () OVER (ORDER BY ts_rank(to_tsvector('english', content), to_tsquery('{fts_search_token}')) desc) as fts_rank,  *
  FROM video_embeddings
  WHERE to_tsvector('english', content) @@ to_tsquery('{fts_search_token}')
  ORDER BY ts_rank_score desc
  LIMIT {top_k * 2}
) SELECT
    COALESCE((1.0 / ({rrf_k} + vector_search.vector_rank)), 0.0) + COALESCE((1.0 / ({rrf_k} + fts_search.fts_rank)), 0.0) AS rrf_score,
    CASE
      WHEN vector_rank IS NOT NULL THEN 'vector'
      WHEN ts_rank_score IS NOT NULL THEN 'fts'
    END AS result_type,
    COALESCE(vector_search.id, fts_search.id) AS id,
    COALESCE(vector_search.content, fts_search.content) AS content,
    COALESCE(vector_search.uri, fts_search.uri) AS uri,
    COALESCE(vector_search.metadata_json, fts_search.metadata_json) AS metadata_json
FROM vector_search
FULL OUTER JOIN fts_search ON vector_search.id = fts_search.id
LEFT JOIN video_embeddings ve ON ve.id = fts_search.id
ORDER BY rrf_score DESC
LIMIT {top_k}
"""

result = await run_query(sql)
result